# Price Elasticity with Rideshare

In [ ]:
# Import packages
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from pathlib import Path
from config import YELLOW_CLEAN05_PARQUET, HVFHV_CLEAN05_PARQUET, HVFHV_CLEAN005_PARQUET, OUTPUT_DIR

In [2]:
yellow = pq.read_table(YELLOW_CLEAN05_PARQUET).to_pandas()
hvfhv = pq.read_table(HVFHV_CLEAN05_PARQUET).to_pandas()

In [3]:
# Calculate driver pay per mile and per minute
hvfhv["driverpay_per_mile"] =  hvfhv["driver_pay"] / hvfhv["trip_miles"]
hvfhv["driverpay_per_min"] =  hvfhv["driver_pay"] / hvfhv["trip_duration_min"]

## Variable creation / aggregation

### Aggregate Yellow Taxi metrics

In [4]:
# Aggregate Yellow Taxi metrics (taxi location ID levels)
yellow_hourly_LocationID = (
    yellow
    .groupby(
        [
            "PU_datetime_hour",
            "PULocationID"
        ],
        as_index=False
    )
    .agg(
        avg_taxi_fare_per_mile_LocationID = ("fare_per_mile", "mean"),
        avg_taxi_fare_per_min_LocationID = ("fare_per_min", "mean"),
        avg_taxi_trip_duration_min_LocationID = ("trip_duration_min", "mean"),
        taxi_rides_LocationID = ("trip_duration_min", "size")
    )
)

In [5]:
# Aggregate Yellow Taxi metrics (borough level)
yellow_hourly_borough = (
    yellow
    .groupby(
        [
            "PU_datetime_hour",
            "PU_Borough"
        ],
        as_index=False
    )
    .agg(
        avg_taxi_fare_per_mile_borough = ("fare_per_mile", "mean"),
        avg_taxi_fare_per_min_borough = ("fare_per_min", "mean"),
        avg_taxi_trip_duration_min_borough = ("trip_duration_min", "mean"),
        taxi_rides_borough = ("trip_duration_min", "size")
    )
)

In [16]:
yellow_hourly_borough

,PU_datetime_hour,PU_Borough,avg_taxi_fare_per_mile_borough,avg_taxi_fare_per_min_borough,avg_taxi_trip_duration_min_borough,taxi_rides_borough
0,2025-01-01 00:00:00,Brooklyn,4.771242,1.445545,15.150000,1
1,2025-01-01 00:00:00,Manhattan,7.997225,1.200334,14.306752,274
2,2025-01-01 00:00:00,Queens,4.127572,2.153633,27.304762,21
3,2025-01-01 01:00:00,Bronx,37.000000,3.415385,1.083333,1
4,2025-01-01 01:00:00,Brooklyn,6.279872,1.284954,12.441667,6
...,...,...,...,...,...,...
30379,2025-12-31 22:00:00,Manhattan,8.541695,1.177951,12.741365,166
30380,2025-12-31 22:00:00,Queens,4.572631,1.903259,19.046078,17
30381,2025-12-31 23:00:00,Brooklyn,9.285714,1.304348,4.983333,1
30382,2025-12-31 23:00:00,Manhattan,7.902738,1.260279,13.117157,136


In [6]:
del yellow

### Created buckets for continuous variables and dummy variables

In [7]:
# Create bucket for daily minimum temperature
hvfhv["tmin_f_bucket"] = pd.qcut(
    hvfhv["tmin_f"],
    q=4,
    labels=[
        "low_temp",
        "mid_low_temp",
        "mid_high_temp",
        "high_temp"
    ],
    duplicates="drop"
)

In [8]:
# Create bucket for rain levels
hvfhv["rain_melted_snow_etc_in"] = pd.to_numeric(
    hvfhv["rain_melted_snow_etc_in"],
    errors="coerce"
)
rain_threshold = hvfhv.loc[
    hvfhv["rain_melted_snow_etc_in"] > 0,
    "rain_melted_snow_etc_in"
].median()

print("Rain threshold:", rain_threshold)

hvfhv["rain_bucket"] = np.select(
    [
        hvfhv["rain_melted_snow_etc_in"] == 0,
        (hvfhv["rain_melted_snow_etc_in"] > 0) &
        (hvfhv["rain_melted_snow_etc_in"] < rain_threshold),
        hvfhv["rain_melted_snow_etc_in"] >= rain_threshold
    ],
    [
        "no_rain",
        "low_rain",
        "high_rain"
    ],
    default="unknown"
)

Rain threshold: 0.09


In [9]:
# Create bucket for shared rides
hvfhv["shared_match_flag"] = (
    hvfhv["shared_match_flag"] == "Y"
).astype(int)

In [10]:
hvfhv.columns

Index(['source', 'request_datetime', 'pickup_datetime', 'dropoff_datetime',
       'PULocationID', 'DOLocationID', 'trip_miles', 'trip_time', 'fare',
       'tolls', 'bcf', 'sales_tax', 'congestion_surcharge', 'airport_fee',
       'tips', 'driver_pay', 'shared_request_flag', 'shared_match_flag',
       'wav_request_flag', 'cbd_congestion_fee', 'trip_duration_min',
       'fare_per_mile', 'fare_per_min', 'take_rate', 'wait_time', 'PU_date',
       'PU_month', 'PU_hour', 'PU_datetime_hour', 'PU_day_name',
       'PU_weekday_num', 'PU_week_category', 'PU_time_zone', 'PU_peak_flag',
       'PU_Borough', 'Zone', 'airport_dummy', 'subway_ridership', 'tmax_f',
       'tmin_f', 'rain_melted_snow_etc_in', 'driverpay_per_mile',
       'driverpay_per_min', 'tmin_f_bucket', 'rain_bucket'],
      dtype='str')

## Uber market share table

In [ ]:
# Merge Yellow Taxi aggregated metrics onto FHVHV
hvfhv = hvfhv.merge(
    yellow_hourly_LocationID,
    on=[
        "PU_datetime_hour",
        "PULocationID"
    ],
    how="left"
)


,source,request_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,fare,tolls,...,tmin_f,rain_melted_snow_etc_in,driverpay_per_mile,driverpay_per_min,tmin_f_bucket,rain_bucket,avg_taxi_fare_per_mile_LocationID,avg_taxi_fare_per_min_LocationID,avg_taxi_trip_duration_min_LocationID,taxi_rides_LocationID
0,uber,2025-01-05 11:33:26,2025-01-05 11:34:58,2025-01-05 11:54:18,163,144,3.450,1160,27.20,0.0,...,28,0.000,4.626087,0.825517,low_temp,no_rain,9.448048,1.122103,14.191667,6.0
1,uber,2025-01-16 22:52:08,2025-01-16 22:55:59,2025-01-16 23:33:40,155,265,19.010,2261,53.35,0.0,...,23,0.001,2.641767,1.332685,low_temp,low_rain,NaN,NaN,NaN,NaN
2,uber,2025-01-19 23:09:59,2025-01-19 23:28:09,2025-01-19 23:41:19,106,14,5.950,790,18.74,0.0,...,24,0.260,2.843697,1.285063,low_temp,high_rain,NaN,NaN,NaN,NaN
3,lyft,2025-01-23 10:50:08,2025-01-23 10:53:50,2025-01-23 11:10:11,180,95,3.467,981,17.85,0.0,...,17,0.000,4.511105,0.956575,low_temp,no_rain,NaN,NaN,NaN,NaN
4,lyft,2025-01-19 01:35:56,2025-01-19 01:44:22,2025-01-19 01:47:21,91,72,0.508,179,8.06,0.0,...,24,0.260,10.767717,1.833520,low_temp,high_rain,NaN,NaN,NaN,NaN


In [ ]:
hvfhv = hvfhv.merge(
    yellow_hourly_borough,
    on=[
        "PU_datetime_hour",
        "PU_Borough"
    ],
    how="left"
)


,source,request_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,fare,tolls,...,tmin_f_bucket,rain_bucket,avg_taxi_fare_per_mile_LocationID,avg_taxi_fare_per_min_LocationID,avg_taxi_trip_duration_min_LocationID,taxi_rides_LocationID,avg_taxi_fare_per_mile_borough,avg_taxi_fare_per_min_borough,avg_taxi_trip_duration_min_borough,taxi_rides_borough
0,uber,2025-01-05 11:33:26,2025-01-05 11:34:58,2025-01-05 11:54:18,163,144,3.450,1160,27.20,0.0,...,low_temp,no_rain,9.448048,1.122103,14.191667,6.0,7.578734,1.306364,11.029983,194.0
1,uber,2025-01-16 22:52:08,2025-01-16 22:55:59,2025-01-16 23:33:40,155,265,19.010,2261,53.35,0.0,...,low_temp,low_rain,NaN,NaN,NaN,NaN,11.833333,1.434046,6.783333,2.0
2,uber,2025-01-19 23:09:59,2025-01-19 23:28:09,2025-01-19 23:41:19,106,14,5.950,790,18.74,0.0,...,low_temp,high_rain,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,lyft,2025-01-23 10:50:08,2025-01-23 10:53:50,2025-01-23 11:10:11,180,95,3.467,981,17.85,0.0,...,low_temp,no_rain,NaN,NaN,NaN,NaN,4.536999,1.369907,34.619444,18.0
4,lyft,2025-01-19 01:35:56,2025-01-19 01:44:22,2025-01-19 01:47:21,91,72,0.508,179,8.06,0.0,...,low_temp,high_rain,NaN,NaN,NaN,NaN,6.440678,1.202109,9.483333,1.0


### Create panel data by pickup datetime hour and location ID

In [ ]:
panel_LocationID = (
    hvfhv
    .groupby(
        ["PU_datetime_hour", "PULocationID", "PU_Borough", "source", # "airport_dummy", "shared_match_flag",
         "PU_day_name", "PU_peak_flag", "PU_time_zone", "subway_ridership", 
         "taxi_rides_LocationID", "avg_taxi_fare_per_mile_LocationID", "avg_taxi_fare_per_min_LocationID", "avg_taxi_trip_duration_min_LocationID",
         "tmin_f_bucket", "rain_bucket"],
        observed=True
    )
    .agg(
        trips=("source", "size"),
        airport_rides=("airport_dummy", "sum"),
        shared_rides=("shared_match_flag", "sum"),
        avg_trip_time=("trip_duration_min", "mean"),
        avg_trip_miles=("trip_miles", "mean"),

        avg_fare_per_mile=("fare_per_mile", "mean"),
        median_fare_per_mile=("fare_per_mile", "median"),
        # q1_fare_per_mile=("fare_per_mile", lambda x: x.quantile(0.25)),
        # q3_fare_per_mile=("fare_per_mile", lambda x: x.quantile(0.75)),

        avg_fare_per_min=("fare_per_min", "mean"),
        median_fare_per_min=("fare_per_min", "median"),
        # q1_fare_per_min=("fare_per_min", lambda x: x.quantile(0.25)),
        # q3_fare_per_min=("fare_per_min", lambda x: x.quantile(0.75)),
        
        avg_driverpay_per_mile=("driverpay_per_mile", "mean"),
        median_driverpay_per_mile=("driverpay_per_mile", "median"),
        # q1_driverpay_per_mile=("driverpay_per_mile", lambda x: x.quantile(0.25)),
        # q3_driverpay_per_mile=("driverpay_per_mile", lambda x: x.quantile(0.75)),
        
        avg_driverpay_per_min=("driverpay_per_min", "mean"),
        median_driverpay_per_min=("driverpay_per_min", "median"),
        # q1_driverpay_per_min=("driverpay_per_min", lambda x: x.quantile(0.25)),
        # q3_driverpay_per_min=("driverpay_per_min", lambda x: x.quantile(0.75)),

        avg_take_rate=("take_rate", "mean"),
        median_take_rate=("take_rate", "median"),
        # q1_take_rate=("take_rate", lambda x: x.quantile(0.25)),
        # q3_take_rate=("take_rate", lambda x: x.quantile(0.75)),
        
        avg_wait_time=("wait_time", "mean"),
        median_wait_time=("wait_time", "median"),
        # q1_wait_rate=("wait_rate", lambda x: x.quantile(0.25)),
        # q3_wait_rate=("wait_rate", lambda x: x.quantile(0.75)),
        
    )
    .reset_index()
)

In [34]:
## Calculate market share
market_totals_LocationID = (
    panel_LocationID
    .groupby(["PU_datetime_hour", "PULocationID"])["trips"]
    .sum()
    .reset_index(name="total_hvfhv_trips")
)

panel_LocationID = panel_LocationID.merge(
    market_totals_LocationID,
    on=["PU_datetime_hour", "PULocationID"],
    how="left"
)

panel_LocationID["hvfhv_market_share"] = (
    panel_LocationID["trips"] / panel_LocationID["total_hvfhv_trips"]
)

panel_LocationID["total_trips"] = 20*panel_LocationID["taxi_rides_LocationID"] + 20*panel_LocationID["total_hvfhv_trips"] + panel_LocationID["subway_ridership"] 
panel_LocationID["total_market_share"] = (
    20*panel_LocationID["trips"] / panel_LocationID["total_trips"]
)


In [28]:
panel_LocationID.to_csv("panel_LocationID.csv",index=False)
panel_LocationID

,PU_datetime_hour,PULocationID,PU_Borough,source,PU_day_name,PU_peak_flag,PU_time_zone,subway_ridership,taxi_rides_LocationID,avg_taxi_fare_per_mile_LocationID,...,avg_driverpay_per_min,median_driverpay_per_min,avg_take_rate,median_take_rate,avg_wait_time,median_wait_time,total_hvfhv_trips,hvfhv_market_share,total_trips,total_market_share
0,2025-01-01 03:00:00,4,Manhattan,lyft,Wednesday,0,Late Night,11242.0,2.0,6.746193,...,1.349498,1.349498,0.112479,0.112479,7.716667,7.716667,8,0.250000,11442.0,0.003496
1,2025-01-01 03:00:00,4,Manhattan,uber,Wednesday,0,Late Night,11242.0,2.0,6.746193,...,2.549284,1.740682,0.014849,0.054130,5.891667,5.758333,8,0.750000,11442.0,0.010488
2,2025-01-01 03:00:00,7,Queens,lyft,Wednesday,0,Late Night,2007.0,1.0,4.263715,...,1.325930,1.359982,0.237196,0.265347,4.600000,4.358333,32,0.250000,2667.0,0.059993
3,2025-01-01 03:00:00,7,Queens,uber,Wednesday,0,Late Night,2007.0,1.0,4.263715,...,1.379051,1.317728,0.023803,0.025873,6.207639,5.666667,32,0.750000,2667.0,0.179978
4,2025-01-01 03:00:00,24,Manhattan,lyft,Wednesday,0,Late Night,11242.0,1.0,6.825397,...,0.801486,0.801486,0.429206,0.429206,7.033333,7.033333,2,0.500000,11302.0,0.001770
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
698441,2025-12-31 23:00:00,261,Manhattan,uber,Wednesday,0,Evening,27268.0,2.0,8.247577,...,1.247925,1.202358,0.363282,0.375430,7.328571,6.850000,10,0.700000,27508.0,0.005089
698442,2025-12-31 23:00:00,262,Manhattan,lyft,Wednesday,0,Evening,27268.0,1.0,5.431235,...,1.155716,1.155716,0.082702,0.082702,4.300000,4.300000,3,0.333333,27348.0,0.000731
698443,2025-12-31 23:00:00,262,Manhattan,uber,Wednesday,0,Evening,27268.0,1.0,5.431235,...,1.315610,1.315610,0.356957,0.356957,5.516667,5.516667,3,0.666667,27348.0,0.001463
698444,2025-12-31 23:00:00,263,Manhattan,lyft,Wednesday,0,Evening,27268.0,6.0,6.192858,...,1.001952,1.001952,0.208382,0.208382,5.166667,5.166667,6,0.333333,27508.0,0.001454


### Create panel data by pickup datetime hour and borough

In [ ]:
panel_borough = (
    hvfhv
    .groupby(
        ["PU_datetime_hour", "PU_Borough", "source", # "airport_dummy", "shared_match_flag",
         "PU_day_name", "PU_peak_flag", "PU_time_zone", "subway_ridership", 
         "taxi_rides_borough", "avg_taxi_fare_per_mile_borough", "avg_taxi_fare_per_min_borough", "avg_taxi_trip_duration_min_borough",
         "tmin_f_bucket", "rain_bucket"],
        observed=True
    )
    .agg(
        trips=("source", "size"),
        airport_rides=("airport_dummy", "sum"),
        shared_rides=("shared_match_flag", "sum"),
        avg_trip_time=("trip_duration_min", "mean"),
        avg_trip_miles=("trip_miles", "mean"),

        avg_fare_per_mile=("fare_per_mile", "mean"),
        median_fare_per_mile=("fare_per_mile", "median"),
        q1_fare_per_mile=("fare_per_mile", lambda x: x.quantile(0.25)),
        q3_fare_per_mile=("fare_per_mile", lambda x: x.quantile(0.75)),

        avg_fare_per_min=("fare_per_min", "mean"),
        median_fare_per_min=("fare_per_min", "median"),
        q1_fare_per_min=("fare_per_min", lambda x: x.quantile(0.25)),
        q3_fare_per_min=("fare_per_min", lambda x: x.quantile(0.75)),
        
        avg_driverpay_per_mile=("driverpay_per_mile", "mean"),
        median_driverpay_per_mile=("driverpay_per_mile", "median"),
        q1_driverpay_per_mile=("driverpay_per_mile", lambda x: x.quantile(0.25)),
        q3_driverpay_per_mile=("driverpay_per_mile", lambda x: x.quantile(0.75)),
        
        avg_driverpay_per_min=("driverpay_per_min", "mean"),
        median_driverpay_per_min=("driverpay_per_min", "median"),
        q1_driverpay_per_min=("driverpay_per_min", lambda x: x.quantile(0.25)),
        q3_driverpay_per_min=("driverpay_per_min", lambda x: x.quantile(0.75)),

        avg_take_rate=("take_rate", "mean"),
        median_take_rate=("take_rate", "median"),
        q1_take_rate=("take_rate", lambda x: x.quantile(0.25)),
        q3_take_rate=("take_rate", lambda x: x.quantile(0.75)),
        
        avg_wait_time=("wait_time", "mean"),
        median_wait_time=("wait_time", "median"),
        q1_wait_time=("wait_time", lambda x: x.quantile(0.25)),
        q3_wait_time=("wait_time", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)


In [49]:
panel_borough = panel_borough.drop(['total_hvfhv_trips',
       'market_share_among_hvfhv', 'total_trips', 'total_market_share',
       'hvfhv_market_share', 'taxi_market_share', 'subway_market_share'], axis=1)
panel_borough

,PU_datetime_hour,PU_Borough,source,PU_day_name,PU_peak_flag,PU_time_zone,subway_ridership,taxi_rides_borough,avg_taxi_fare_per_mile_borough,avg_taxi_fare_per_min_borough,...,q1_driverpay_per_min,q3_driverpay_per_min,avg_take_rate,median_take_rate,q1_take_rate,q3_take_rate,avg_wait_time,median_wait_time,q1_wait_rate,q3_wait_rate
0,2025-01-01 03:00:00,Manhattan,lyft,Wednesday,0,Late Night,11242.0,3200.0,7.596356,1.455695,...,1.058633,1.428483,0.193975,0.198541,0.091009,0.331738,5.457815,4.783333,3.500000,7.050000
1,2025-01-01 03:00:00,Manhattan,uber,Wednesday,0,Late Night,11242.0,3200.0,7.596356,1.455695,...,1.262720,1.889379,0.043477,0.087276,-0.088739,0.235055,6.924463,5.800000,3.941667,8.766667
2,2025-01-01 03:00:00,Queens,lyft,Wednesday,0,Late Night,2007.0,80.0,5.998923,2.467161,...,1.107282,1.412488,0.188567,0.197358,0.066205,0.296948,6.422680,5.600000,4.266667,8.466667
3,2025-01-01 03:00:00,Queens,uber,Wednesday,0,Late Night,2007.0,80.0,5.998923,2.467161,...,1.074354,1.514778,0.072585,0.118850,-0.052239,0.258396,7.325623,6.550000,4.983333,9.216667
4,2025-01-01 04:00:00,Brooklyn,lyft,Wednesday,0,Late Night,3312.0,60.0,4.898812,1.395050,...,1.005118,1.300902,0.166986,0.189663,0.072798,0.283398,5.377134,4.750000,3.679167,6.450000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55149,2025-12-31 23:00:00,Brooklyn,uber,Wednesday,0,Evening,10263.0,20.0,9.285714,1.304348,...,0.887254,1.113207,0.170039,0.186635,0.063104,0.313447,5.079100,4.358333,2.920833,6.095833
55150,2025-12-31 23:00:00,Manhattan,lyft,Wednesday,0,Evening,27268.0,2720.0,7.902738,1.260279,...,0.848594,1.120082,0.241009,0.261972,0.153938,0.377569,5.405241,4.333333,2.887500,6.991667
55151,2025-12-31 23:00:00,Manhattan,uber,Wednesday,0,Evening,27268.0,2720.0,7.902738,1.260279,...,0.929664,1.293712,0.229619,0.249168,0.114837,0.395133,4.462097,3.683333,2.516667,5.700000
55152,2025-12-31 23:00:00,Queens,lyft,Wednesday,0,Evening,5393.0,240.0,4.033166,2.042565,...,0.982239,1.281426,0.198719,0.210115,0.122510,0.303404,5.558537,5.383333,3.408333,7.033333


In [50]:
## Calculate market share
market_totals_borough = (
    panel_borough
    .groupby(["PU_datetime_hour", "PU_Borough"])["trips"]
    .sum()
    .reset_index(name="total_hvfhv_trips")
)

panel_borough = panel_borough.merge(
    market_totals_borough,
    on=["PU_datetime_hour", "PU_Borough"],
    how="left"
)
panel_borough["market_share_among_hvfhv"] = (
    panel_borough["trips"] / panel_borough["total_hvfhv_trips"]
)

panel_borough["total_trips"] = panel_borough["subway_ridership"] + 20*panel_borough["taxi_rides_borough"] + 20*panel_borough["total_hvfhv_trips"]
panel_borough["total_market_share"] = (
    20*panel_borough["trips"] / panel_borough["total_trips"]
)

panel_borough["hvfhv_market_share"] = panel_borough["total_hvfhv_trips"]*20 / panel_borough["total_trips"]
panel_borough["taxi_rides_borough"] = panel_borough["taxi_rides_borough"]*20
panel_borough["trips"] = panel_borough["trips"]*20
panel_borough["total_hvfhv_trips"] = panel_borough["total_hvfhv_trips"]*20
panel_borough["taxi_market_share"] = panel_borough["taxi_rides_borough"] / panel_borough["total_trips"]
panel_borough["subway_market_share"] = panel_borough["subway_ridership"] / panel_borough["total_trips"]

In [52]:
panel_borough.to_csv("panel_borough.csv",index=False)
panel_borough

,PU_datetime_hour,PU_Borough,source,PU_day_name,PU_peak_flag,PU_time_zone,subway_ridership,taxi_rides_borough,avg_taxi_fare_per_mile_borough,avg_taxi_fare_per_min_borough,...,median_wait_time,q1_wait_rate,q3_wait_rate,total_hvfhv_trips,market_share_among_hvfhv,total_trips,total_market_share,hvfhv_market_share,taxi_market_share,subway_market_share
0,2025-01-01 03:00:00,Manhattan,lyft,Wednesday,0,Late Night,11242.0,64000.0,7.596356,1.455695,...,4.783333,3.500000,7.050000,288000,0.245833,363242.0,0.194911,0.792860,0.176191,0.030949
1,2025-01-01 03:00:00,Manhattan,uber,Wednesday,0,Late Night,11242.0,64000.0,7.596356,1.455695,...,5.800000,3.941667,8.766667,288000,0.754167,363242.0,0.597948,0.792860,0.176191,0.030949
2,2025-01-01 03:00:00,Queens,lyft,Wednesday,0,Late Night,2007.0,1600.0,5.998923,2.467161,...,5.600000,4.266667,8.466667,183200,0.211790,186807.0,0.207701,0.980691,0.008565,0.010744
3,2025-01-01 03:00:00,Queens,uber,Wednesday,0,Late Night,2007.0,1600.0,5.998923,2.467161,...,6.550000,4.983333,9.216667,183200,0.788210,186807.0,0.772990,0.980691,0.008565,0.010744
4,2025-01-01 04:00:00,Brooklyn,lyft,Wednesday,0,Late Night,3312.0,1200.0,4.898812,1.395050,...,4.750000,3.679167,6.450000,214000,0.306542,218512.0,0.300212,0.979351,0.005492,0.015157
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55149,2025-12-31 23:00:00,Brooklyn,uber,Wednesday,0,Evening,10263.0,400.0,9.285714,1.304348,...,4.358333,2.920833,6.095833,354800,0.701240,365463.0,0.680780,0.970823,0.001095,0.028082
55150,2025-12-31 23:00:00,Manhattan,lyft,Wednesday,0,Evening,27268.0,54400.0,7.902738,1.260279,...,4.333333,2.887500,6.991667,255600,0.303599,337268.0,0.230084,0.757854,0.161296,0.080850
55151,2025-12-31 23:00:00,Manhattan,uber,Wednesday,0,Evening,27268.0,54400.0,7.902738,1.260279,...,3.683333,2.516667,5.700000,255600,0.696401,337268.0,0.527770,0.757854,0.161296,0.080850
55152,2025-12-31 23:00:00,Queens,lyft,Wednesday,0,Evening,5393.0,4800.0,4.033166,2.042565,...,5.383333,3.408333,7.033333,180400,0.272727,190593.0,0.258142,0.946520,0.025185,0.028296


In [70]:
panel_borough = panel_borough.rename(columns={'q1_wait_rate': 'q1_wait_time','q3_wait_rate': 'q3_wait_time'})

### 1. Market share model

Uber gains share during urgency/high-price conditions.
uber_share ~ fare_per_mile + peak + airport + weather

In [72]:
# Merge lyft features to uber df
uber_df = panel_borough[
    panel_borough['source'] == 'uber'
].copy()

lyft_df = panel_borough[
    panel_borough['source'] == 'lyft'
].copy()

# Add prefix to lyft and uber dfs
cols = [ 'airport_rides','shared_rides','avg_trip_time','avg_trip_miles','trips',
        'avg_fare_per_mile', 'median_fare_per_mile','q1_fare_per_mile','q3_fare_per_mile',
        'avg_driverpay_per_mile', 'median_driverpay_per_mile','q1_driverpay_per_mile','q3_driverpay_per_mile', 
        'avg_take_rate','median_take_rate','q1_take_rate', 'q3_take_rate',
        'avg_wait_time','median_wait_time','q1_wait_time','q3_wait_time']
rename_map = {c: f"lyft_{c}" for c in cols if c in lyft_df.columns} 
lyft_df = lyft_df.rename(columns=rename_map)
rename_map = {c: f"uber_{c}" for c in cols if c in uber_df.columns} 
uber_df = uber_df.rename(columns=rename_map)

uber_df = uber_df.merge(
    lyft_df[['PU_datetime_hour', 'PU_Borough', 'lyft_airport_rides','lyft_shared_rides','lyft_avg_trip_time','lyft_avg_trip_miles','lyft_trips',
        'lyft_avg_fare_per_mile', 'lyft_median_fare_per_mile','lyft_q1_fare_per_mile','lyft_q3_fare_per_mile',
        'lyft_avg_driverpay_per_mile', 'lyft_median_driverpay_per_mile','lyft_q1_driverpay_per_mile','lyft_q3_driverpay_per_mile', 
        'lyft_avg_take_rate','lyft_median_take_rate','lyft_q1_take_rate', 'lyft_q3_take_rate',
        'lyft_avg_wait_time','lyft_median_wait_time','lyft_q1_wait_time','lyft_q3_wait_time']],
    on=[
        'PU_datetime_hour',
        'PU_Borough'
    ],
    how='left'
)


In [79]:
uber_df.columns

Index(['PU_datetime_hour', 'PU_Borough', 'source', 'PU_day_name',
       'PU_peak_flag', 'PU_time_zone', 'subway_ridership',
       'taxi_rides_borough', 'avg_taxi_fare_per_mile_borough',
       'avg_taxi_fare_per_min_borough', 'avg_taxi_trip_duration_min_borough',
       'tmin_f_bucket', 'rain_bucket', 'uber_trips', 'uber_airport_rides',
       'uber_shared_rides', 'uber_avg_trip_time', 'uber_avg_trip_miles',
       'uber_avg_fare_per_mile', 'uber_median_fare_per_mile',
       'uber_q1_fare_per_mile', 'uber_q3_fare_per_mile', 'avg_fare_per_min',
       'median_fare_per_min', 'q1_fare_per_min', 'q3_fare_per_min',
       'uber_avg_driverpay_per_mile', 'uber_median_driverpay_per_mile',
       'uber_q1_driverpay_per_mile', 'uber_q3_driverpay_per_mile',
       'avg_driverpay_per_min', 'median_driverpay_per_min',
       'q1_driverpay_per_min', 'q3_driverpay_per_min', 'uber_avg_take_rate',
       'uber_median_take_rate', 'uber_q1_take_rate', 'uber_q3_take_rate',
       'uber_avg_wait_time', 

In [107]:
uber_df

,PU_datetime_hour,PU_Borough,source,PU_day_name,PU_peak_flag,PU_time_zone,subway_ridership,taxi_rides_borough,avg_taxi_fare_per_mile_borough,avg_taxi_fare_per_min_borough,...,PU_week,PU_hour,uber_revenue,lyft_revenue,uber_revenue_per_trip,lyft_revenue_per_trip,lag_rel_driver_pay_per_mile,log_total_hvfhv_trips,lag_log_total_hvfhv,lag_log_rel_lyft_take_rate
14,2025-01-01 09:00:00,Bronx,uber,Wednesday,1,Morning,5169.0,400.0,8.285714,1.851064,...,2024-12-29,9,1.095941e+05,33257.803038,4.348974,3.779296,NaN,10.434145,NaN,NaN
19,2025-01-01 11:00:00,Bronx,uber,Wednesday,0,Morning,5181.0,400.0,5.000000,1.551220,...,2024-12-29,11,1.687368e+05,64700.852483,5.021929,5.990820,-0.194730,10.701017,10.434145,-0.169502
23,2025-01-01 12:00:00,Bronx,uber,Wednesday,0,Afternoon,5769.0,400.0,3.881579,0.604715,...,2024-12-29,12,1.639087e+05,44737.308560,4.997218,5.083785,-0.027719,10.635879,10.701017,-0.488253
29,2025-01-01 14:00:00,Bronx,uber,Wednesday,0,Afternoon,8597.0,1200.0,4.241603,1.096191,...,2024-12-29,14,1.744084e+05,68821.918116,3.406413,6.617492,-0.190688,11.028433,10.635879,-0.441304
63,2025-01-02 03:00:00,Bronx,uber,Thursday,0,Late Night,1120.0,400.0,3.549383,1.263736,...,2024-12-29,3,8.881948e+04,16276.471695,6.530844,5.086397,-0.016436,9.729194,11.028433,-0.358898
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27551,2025-12-31 19:00:00,Queens,uber,Wednesday,1,Evening,16683.0,7200.0,3.899971,1.697740,...,2025-12-28,19,9.016655e+05,297562.608811,4.858111,4.250894,0.018091,12.451373,12.349311,0.098966
27555,2025-12-31 20:00:00,Queens,uber,Wednesday,0,Evening,14854.0,9600.0,4.239633,1.986942,...,2025-12-28,20,1.022723e+06,395906.562098,4.824164,5.236859,0.004340,12.569329,12.451373,0.100532
27559,2025-12-31 21:00:00,Queens,uber,Wednesday,0,Evening,12387.0,8400.0,6.079036,1.930957,...,2025-12-28,21,1.415786e+06,457265.672060,6.872749,6.315824,-0.068871,12.536818,12.569329,-0.013881
27563,2025-12-31 22:00:00,Queens,uber,Wednesday,0,Evening,11207.0,6800.0,4.572631,1.903259,...,2025-12-28,22,1.273015e+06,516820.678569,6.616500,6.909367,-0.037042,12.495756,12.536818,0.207005


In [92]:
# Derive relative measures and perform transformation

## Demand; price elasticity
# fare per mile vs lyft
uber_df['log_rel_lyft_fare_per_mile'] = np.log(
    uber_df['uber_median_fare_per_mile']
    / uber_df['lyft_median_fare_per_mile']
)

uber_df['log_rel_lyft_q1_fare_per_mile'] = np.log(
    uber_df['uber_q1_fare_per_mile']
    / uber_df['lyft_q1_fare_per_mile']
)

uber_df['log_rel_lyft_q3_fare_per_mile'] = np.log(
    uber_df['uber_q3_fare_per_mile']
    / uber_df['lyft_q3_fare_per_mile']
)

# fare per mile vs taxi
uber_df['log_rel_taxi_fare_per_mile'] = np.log(
    uber_df['uber_median_fare_per_mile']
    / uber_df['avg_taxi_fare_per_mile_borough']
)

uber_df['log_rel_taxi_q1_fare_per_mile'] = np.log(
    uber_df['uber_q1_fare_per_mile']
    / uber_df['avg_taxi_fare_per_mile_borough']
)

uber_df['log_rel_taxi_q3_fare_per_mile'] = np.log(
    uber_df['uber_q3_fare_per_mile']
    / uber_df['avg_taxi_fare_per_mile_borough']
)

## Supply
# wait time vs lyft
uber_df['log_rel_lyft_wait_time'] = np.log(
    uber_df['uber_median_wait_time']
    / uber_df['lyft_median_wait_time']
)

uber_df['log_rel_lyft_q3_wait_time'] = np.log(
    uber_df['uber_q3_wait_time']
    / uber_df['lyft_q3_wait_time']
)

uber_df['log_rel_lyft_q1_wait_time'] = np.log(
    uber_df['uber_q1_wait_time']
    / uber_df['lyft_q1_wait_time']
)

# driver pay per mile vs lyft
uber_df['log_rel_lyft_driverpay_per_mile'] = np.log(
    uber_df['uber_median_driverpay_per_mile']
    / uber_df['lyft_median_driverpay_per_mile']
)

uber_df['log_rel_lyft_q3_driverpay_per_mile'] = np.log(
    uber_df['uber_q3_driverpay_per_mile']
    / uber_df['lyft_q3_driverpay_per_mile']
)

uber_df['log_rel_lyft_q1_driverpay_per_mile'] = np.log(
    uber_df['uber_q1_driverpay_per_mile']
    / uber_df['lyft_q1_driverpay_per_mile']
)

# take rate vs lyft
uber_df['log_rel_lyft_take_rate'] = np.log(
    uber_df['uber_median_take_rate']
    / uber_df['lyft_median_take_rate']
)

# shared rides rate
uber_df['uber_shared_rides_share'] = (
    uber_df['uber_shared_rides'] * 20
    / uber_df['uber_trips']
)
uber_df['lyft_shared_rides_share'] = (
    uber_df['lyft_shared_rides'] * 20
    / uber_df['lyft_trips']
)

uber_df['rel_lyft_shared'] = (
    uber_df['uber_shared_rides_share']
    - uber_df['lyft_shared_rides_share']
)

# airports
uber_df['uber_airport_rides_share'] = (
    uber_df['uber_airport_rides'] * 20
    / uber_df['uber_trips']
)
uber_df['lyft_airport_rides_share'] = (
    uber_df['lyft_airport_rides'] * 20
    / uber_df['lyft_trips']
)

uber_df['rel_lyft_airport'] = (
    uber_df['uber_airport_rides_share']
    - uber_df['lyft_airport_rides_share']
)

uber_df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)


,PU_datetime_hour,PU_Borough,source,PU_day_name,PU_peak_flag,PU_time_zone,subway_ridership,taxi_rides_borough,avg_taxi_fare_per_mile_borough,avg_taxi_fare_per_min_borough,...,uber_airport_rides_share,lyft_airport_rides_share,log_rel_lyft_airport,rel_lyft_shared,rel_lyft_airport,log_rel_taxi_q1_fare_per_mile,uber_share_logit,log_rel_lyft_q3_driverpay_per_mile,log_rel_lyft_q1_driverpay_per_mile,log_rel_lyft_q1_wait_time
0,2025-01-01 03:00:00,Manhattan,uber,Wednesday,0,Late Night,11242.0,64000.0,7.596356,1.455695,...,0.000000,0.000000,NaN,0.002578,0.000000,-0.557695,0.396924,0.215416,0.030901,0.118841
1,2025-01-01 03:00:00,Queens,uber,Wednesday,0,Late Night,2007.0,1600.0,5.998923,2.467161,...,0.000277,0.000000,NaN,0.003778,0.000277,-0.499129,1.225271,0.039748,-0.019212,0.155266
2,2025-01-01 04:00:00,Brooklyn,uber,Wednesday,0,Late Night,3312.0,1200.0,4.898812,1.395050,...,0.000000,0.000000,NaN,0.002965,0.000000,-0.031554,0.749816,0.126738,0.005101,0.066801
3,2025-01-01 04:00:00,Manhattan,uber,Wednesday,0,Late Night,7101.0,32400.0,7.117200,1.452731,...,0.000000,0.000000,NaN,0.003459,0.000000,-0.611524,0.241580,0.155286,0.055212,0.256007
4,2025-01-01 04:00:00,Queens,uber,Wednesday,0,Late Night,2481.0,800.0,5.959559,1.249056,...,0.000385,0.000000,NaN,0.002885,0.000385,-0.508775,1.103980,-0.251659,-0.132233,0.306374
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27572,2025-12-31 22:00:00,Manhattan,uber,Wednesday,0,Evening,38399.0,66400.0,8.541695,1.177951,...,0.000000,0.000000,NaN,0.001295,0.000000,-0.440657,0.064198,0.094614,0.037955,-0.161544
27573,2025-12-31 22:00:00,Queens,uber,Wednesday,0,Evening,11207.0,6800.0,4.572631,1.903259,...,0.002183,0.004545,-0.733450,0.001975,-0.002363,-0.080567,0.729053,0.031671,-0.017151,0.163629
27574,2025-12-31 23:00:00,Brooklyn,uber,Wednesday,0,Evening,10263.0,400.0,9.285714,1.304348,...,0.000000,0.000000,NaN,0.000937,0.000000,-0.532824,0.757358,-0.043484,-0.017578,-0.080811
27575,2025-12-31 23:00:00,Manhattan,uber,Wednesday,0,Evening,27268.0,54400.0,7.902738,1.260279,...,0.000000,0.000000,NaN,0.000899,0.000000,-0.497707,0.111195,0.039937,-0.062907,-0.137456


In [94]:

uber_df = uber_df.dropna(
    subset=[
        'log_rel_lyft_fare_per_mile',
        'log_rel_lyft_q3_fare_per_mile',
        'log_rel_lyft_q1_fare_per_mile',
        'log_rel_taxi_fare_per_mile',
        'log_rel_taxi_q3_fare_per_mile',
        'log_rel_taxi_q1_fare_per_mile',
        'log_rel_lyft_wait_time',
        'log_rel_lyft_q3_wait_time',
        'log_rel_lyft_q1_wait_time',
        'log_rel_lyft_driverpay_per_mile',
        'log_rel_lyft_take_rate'
    ]
)

eps = 1e-6

uber_df['uber_share_logit'] = np.log(
    (uber_df['total_market_share'] + eps)
    /
    (1 - uber_df['total_market_share'] + eps)
)


In [27]:
eps = 1e-6

uber_df['uber_share_among_hvfhv_logit'] = np.log(
    (uber_df['market_share_among_hvfhv'] + eps)
    /
    (1 - uber_df['market_share_among_hvfhv'] + eps)
)


In [26]:
uber_df['lyft_share'] = uber_df['lyft_trips'] / uber_df['total_trips']

In [ ]:
uber_df['PU_date'] = uber_df['PU_datetime_hour'].dt.date

AttributeError: Can only use .dt accessor with datetimelike values

In [31]:
dt = pd.to_datetime(uber_df["PU_date"], errors="coerce") 
uber_df["PU_week"] = (dt - pd.to_timedelta((dt.dt.weekday + 1) % 7, unit="D")).dt.date

In [9]:
weekday_list = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"] 
uber_df["PU_weekday"] = uber_df["PU_day_name"].str.strip().str.title().isin(weekday_list).astype(int)

In [34]:
uber_df["PU_datetime_hour"] = pd.to_datetime(uber_df["PU_datetime_hour"], errors="coerce")
uber_df["PU_hour"] = uber_df["PU_datetime_hour"].dt.hour

In [39]:
uber_df['uber_revenue_per_trip'] = uber_df['uber_avg_trip_miles']*uber_df['uber_avg_fare_per_mile']*uber_df['uber_avg_take_rate']
uber_df['lyft_revenue_per_trip'] = uber_df['lyft_avg_trip_miles']*uber_df['lyft_avg_fare_per_mile']*uber_df['lyft_avg_take_rate']
uber_df['uber_revenue'] = uber_df['uber_revenue_per_trip']*uber_df['uber_trips']
uber_df['lyft_revenue'] = uber_df['lyft_revenue_per_trip']*uber_df['lyft_trips']

In [40]:
uber_df.to_csv("uber_panel.csv",index=False)

## Price Elasticity & Regression Analysis

In [41]:
uber_df = pd.read_csv("uber_panel.csv")
uber_df

,PU_datetime_hour,PU_Borough,source,PU_day_name,PU_peak_flag,PU_time_zone,subway_ridership,taxi_rides_borough,avg_taxi_fare_per_mile_borough,avg_taxi_fare_per_min_borough,...,PU_date,PU_weekday,lyft_share,uber_share_among_hvfhv_logit,PU_week,PU_hour,uber_revenue,lyft_revenue,uber_revenue_per_trip,lyft_revenue_per_trip
0,2025-01-01 03:00:00,Manhattan,uber,Wednesday,0,Late Night,11242.0,64000.0,7.596356,1.455695,...,2025-01-01,1,0.194911,1.120957,2024-12-29,3,5.002334e+05,625717.451300,2.303100,8.837817
1,2025-01-01 03:00:00,Queens,uber,Wednesday,0,Late Night,2007.0,1600.0,5.998923,2.467161,...,2025-01-01,1,0.207701,1.314164,2024-12-29,3,4.102376e+05,260680.145663,2.840981,6.718560
2,2025-01-01 04:00:00,Brooklyn,uber,Wednesday,0,Late Night,3312.0,1200.0,4.898812,1.395050,...,2025-01-01,1,0.300212,0.816334,2024-12-29,4,5.084679e+05,382052.423599,3.426334,5.823970
3,2025-01-01 04:00:00,Manhattan,uber,Wednesday,0,Late Night,7101.0,32400.0,7.117200,1.452731,...,2025-01-01,1,0.265961,0.744770,2024-12-29,4,7.081192e+05,346155.831189,5.566975,5.731057
4,2025-01-01 04:00:00,Queens,uber,Wednesday,0,Late Night,2481.0,800.0,5.959559,1.249056,...,2025-01-01,1,0.225302,1.203970,2024-12-29,4,6.089844e+05,151058.404014,5.855619,4.841616
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27562,2025-12-31 22:00:00,Manhattan,uber,Wednesday,0,Evening,38399.0,66400.0,8.541695,1.177951,...,2025-12-31,1,0.250446,0.722946,2025-12-28,22,2.761060e+06,963821.794471,11.921675,8.574927
27563,2025-12-31 22:00:00,Queens,uber,Wednesday,0,Evening,11207.0,6800.0,4.572631,1.903259,...,2025-12-31,1,0.262266,0.944756,2025-12-28,22,1.273015e+06,516820.678569,6.616500,6.909367
27564,2025-12-31 23:00:00,Brooklyn,uber,Wednesday,0,Evening,10263.0,400.0,9.285714,1.304348,...,2025-12-31,1,0.290043,0.853208,2025-12-28,23,1.176541e+06,605536.138890,4.728862,5.712605
27565,2025-12-31 23:00:00,Manhattan,uber,Wednesday,0,Evening,27268.0,54400.0,7.902738,1.260279,...,2025-12-31,1,0.230084,0.830214,2025-12-28,23,1.675464e+06,636130.343734,9.412718,8.197556


### Demand Elasticity (High-level Regression)

In [133]:
base_controls = """
+ C(PU_weekday)
+ C(PU_time_zone)
+ C(rain_bucket)
+ C(tmin_f_bucket)
+ C(PU_Borough)
"""


In [134]:
m1 = smf.ols(
    formula=f"""
    uber_share_logit
    ~ log_rel_lyft_fare_per_mile
    {base_controls}
    """,
    data=uber_df
).fit(cov_type='HC3')

print(m1.summary())


                            OLS Regression Results                            
Dep. Variable:       uber_share_logit   R-squared:                       0.681
Model:                            OLS   Adj. R-squared:                  0.681
Method:                 Least Squares   F-statistic:                     4261.
Date:                Thu, 07 May 2026   Prob (F-statistic):               0.00
Time:                        04:18:33   Log-Likelihood:                 4415.0
No. Observations:               27567   AIC:                            -8802.
Df Residuals:                   27553   BIC:                            -8687.
Df Model:                          13                                         
Covariance Type:                  HC3                                         
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

In [135]:
summary_text_path = OUTPUT_DIR / "m1_summary.txt" 
with open(summary_text_path, "w") as f: 
    f.write(m1.summary().as_text())

In [129]:
m2 = smf.ols(
    formula=f"""
    uber_share_logit
    ~ log_rel_lyft_fare_per_mile
    + log_rel_lyft_take_rate
    + log_rel_lyft_fare_per_mile * C(PU_time_zone)
    + log_rel_lyft_fare_per_mile * C(PU_Borough)
    + log_rel_lyft_fare_per_mile * C(rain_bucket)
    """,
    data=uber_df
).fit(cov_type='HC3')

print(m2.summary())

                            OLS Regression Results                            
Dep. Variable:       uber_share_logit   R-squared:                       0.673
Model:                            OLS   Adj. R-squared:                  0.673
Method:                 Least Squares   F-statistic:                     4021.
Date:                Thu, 07 May 2026   Prob (F-statistic):               0.00
Time:                        03:13:49   Log-Likelihood:                 4062.8
No. Observations:               27567   AIC:                            -8088.
Df Residuals:                   27548   BIC:                            -7931.
Df Model:                          18                                         
Covariance Type:                  HC3                                         
                                                               coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------

In [ ]:
m1 = smf.ols(
    formula=f"""
    uber_share_logit
    ~ log_rel_lyft_fare_per_mile
    {base_controls}
    """,
    data=uber_df
).fit(cov_type='HC3')

print(m1.summary())


In [30]:
m2 = smf.ols(
    formula=f"""
    uber_share_among_hvfhv_logit
    ~ log_rel_lyft_fare_per_mile
    {base_controls}
    """,
    data=uber_df
).fit(cov_type='HC3')

print(m2.summary())


                                 OLS Regression Results                                 
Dep. Variable:     uber_share_among_hvfhv_logit   R-squared:                       0.239
Model:                                      OLS   Adj. R-squared:                  0.239
Method:                           Least Squares   F-statistic:                     549.8
Date:                          Wed, 06 May 2026   Prob (F-statistic):               0.00
Time:                                  18:41:29   Log-Likelihood:                 5455.3
No. Observations:                         27567   AIC:                        -1.088e+04
Df Residuals:                             27553   BIC:                        -1.077e+04
Df Model:                                    13                                         
Covariance Type:                            HC3                                         
                                        coef    std err          z      P>|z|      [0.025      0.975]
--------

In [ ]:
m2 = smf.ols(
    formula=f"""
    uber_share_among_hvfhv_logit
    ~ log_rel_lyft_fare_per_mile
    {base_controls}
    """,
    data=uber_df
).fit(cov_type='HC3')

print(m2.summary())


                                 OLS Regression Results                                 
Dep. Variable:     uber_share_among_hvfhv_logit   R-squared:                       0.239
Model:                                      OLS   Adj. R-squared:                  0.239
Method:                           Least Squares   F-statistic:                     549.8
Date:                          Wed, 06 May 2026   Prob (F-statistic):               0.00
Time:                                  18:41:29   Log-Likelihood:                 5455.3
No. Observations:                         27567   AIC:                        -1.088e+04
Df Residuals:                             27553   BIC:                        -1.077e+04
Df Model:                                    13                                         
Covariance Type:                            HC3                                         
                                        coef    std err          z      P>|z|      [0.025      0.975]
--------

### Demand Elasticity by Sub-segment

In [127]:
def segment_market_analysis(
    df,
    segment_cols,
    y_col='uber_share_logit',
    price_col='log_rel_lyft_fare_per_mile',
    control_cols=None,
    min_obs=100,
    min_price_unique=10
):

    if control_cols is None:
        control_cols = []

    work = df.copy()

    # ----------------------------
    # Create segment key
    # ----------------------------

    work['segment'] = (
        work[segment_cols]
        .astype(str)
        .agg(' | '.join, axis=1)
    )

    results = []

    # ----------------------------
    # Loop by segment
    # ----------------------------

    for segment, g in work.groupby('segment'):

        g = g.copy()

        needed_cols = [y_col, price_col] + control_cols

        g = g.dropna(subset=needed_cols)

        if len(g) < min_obs:
            continue

        if g[price_col].nunique() < min_price_unique:
            continue

        # ----------------------------
        # Regression
        # ----------------------------

        rhs = [price_col] + control_cols

        formula = f"{y_col} ~ " + " + ".join(rhs)

        try:

            model = smf.ols(
                formula=formula,
                data=g
            ).fit(cov_type='HC3')

            coef = model.params.get(price_col, np.nan)

            pval = model.pvalues.get(price_col, np.nan)

            stderr = model.bse.get(price_col, np.nan)

            # ----------------------------
            # Aggregate market metrics
            # ----------------------------

            total_uber_trips = g['uber_trips'].sum()

            total_lyft_trips = g['lyft_trips'].sum()
            total_trips = g['total_trips'].sum()

            total_rideshare_trips = (
                total_uber_trips
                + total_lyft_trips
            )

            # Revenue
            total_uber_revenue = g['uber_revenue'].sum()

            total_lyft_revenue = g['lyft_revenue'].sum()

            total_market_revenue = (
                total_uber_revenue
                + total_lyft_revenue
            )

            # Market share
            uber_market_share = (
                total_uber_trips
                / total_rideshare_trips
                if total_rideshare_trips > 0
                else np.nan
            )

            # Revenue share
            uber_revenue_share = (
                total_uber_revenue
                / total_market_revenue
                if total_market_revenue > 0
                else np.nan
            )
            
            # Market share
            uber_share_among_hvfhv_logit = g['uber_share_among_hvfhv_logit'].mean()
            uber_share_logit = g['uber_share_logit'].mean()

            # Avg metrics
            avg_uber_fare = g['uber_avg_fare_per_mile'].mean()
            avg_uber_take_rate = g['uber_avg_take_rate'].mean()
            avg_uber_wait = g['uber_avg_wait_time'].mean()
            avg_lyft_fare = g['lyft_avg_fare_per_mile'].mean()
            avg_lyft_take_rate = g['lyft_avg_take_rate'].mean()
            avg_lyft_wait = g['lyft_avg_wait_time'].mean()
            log_rel_fare_per_mile = g['log_rel_lyft_fare_per_mile'].mean()

            # ----------------------------
            # Opportunity score
            # ----------------------------

            # stronger elasticity + larger market
            opportunity_score = (
                abs(coef)
                * np.log1p(total_market_revenue)
            )

            results.append({

                # Segment info
                'segment': segment,
                'segment_definition': segment_cols,

                # Regression
                'n_obs': int(model.nobs),
                'price_elasticity_coef': coef,
                'std_err': stderr,
                'p_value': pval,
                'r_squared': model.rsquared,

                # Market structure
                'uber_market_share': uber_market_share,
                'uber_revenue_share': uber_revenue_share,

                # Trips
                'total_uber_trips': total_uber_trips,
                'total_lyft_trips': total_lyft_trips,
                'total_rideshare_trips': total_rideshare_trips,
                'total_trips': total_trips,

                # Revenue
                'total_uber_revenue': total_uber_revenue,
                'total_lyft_revenue': total_lyft_revenue,
                'total_market_revenue': total_market_revenue,
                
                # Market share
                'uber_share_among_hvfhv_logit': uber_share_among_hvfhv_logit, 
                'uber_share_logit': uber_share_logit, 

                # Avg economics
                'avg_uber_fare_per_mile': avg_uber_fare,
                'avg_uber_take_rate': avg_uber_take_rate,
                'avg_uber_wait_time': avg_uber_wait,
                'avg_lyft_fare_per_mile': avg_lyft_fare,
                'avg_lyft_take_rate': avg_lyft_take_rate,
                'avg_lyft_wait_time': avg_lyft_wait,
                'log_rel_fare_per_mile': log_rel_fare_per_mile,

                # Opportunity
                'opportunity_score': opportunity_score

            })

        except Exception as e:

            results.append({
            'segment': segment,
            # Regression placeholders
            'n_obs': np.nan,
            'price_elasticity_coef': np.nan,
            'std_err': np.nan,
            'p_value': np.nan,
            'r_squared': np.nan,
            # Market structure placeholders
            'uber_market_share': np.nan,
            'uber_revenue_share': np.nan,
            # Trips
            'total_uber_trips': np.nan,
            'total_lyft_trips': np.nan,
            'total_rideshare_trips': np.nan,
            # Revenue
            'total_uber_revenue': np.nan,
            'total_lyft_revenue': np.nan,
            'total_market_revenue': np.nan,
            # Avg economics
            'avg_uber_fare_per_mile': np.nan,
            'avg_uber_take_rate': np.nan,
            'avg_uber_wait_time': np.nan,
            # Opportunity
            'opportunity_score': np.nan,
            # error message
            'error': str(e)
        })

    out = pd.DataFrame(results)

    if not out.empty:

        out['significant_10pct'] = (
            out['p_value'] < 0.10
        )

        out['significant_5pct'] = (
            out['p_value'] < 0.05
        )

        out = out.sort_values(
            'opportunity_score',
            ascending=False
        )

    return out


segment_elasticity_analysis = segment_market_analysis(

    df=uber_df,

    segment_cols=[
        'PU_Borough',
        'rain_bucket',
        # 'tmin_f_bucket',
        'PU_time_zone',
        'PU_weekday'
    ],

    y_col= 'uber_share_logit', #'uber_share_among_hvfhv_logit', # 

    price_col='log_rel_lyft_fare_per_mile',

    control_cols=[
        'log_rel_lyft_take_rate'
    ],

    min_obs=80
)



In [128]:
segment_elasticity_analysis.to_csv("segment_elasticity_analysis.csv",index=False)
segment_elasticity_analysis

,segment,segment_definition,n_obs,price_elasticity_coef,std_err,p_value,r_squared,uber_market_share,uber_revenue_share,total_uber_trips,...,avg_uber_fare_per_mile,avg_uber_take_rate,avg_uber_wait_time,avg_lyft_fare_per_mile,avg_lyft_take_rate,avg_lyft_wait_time,log_rel_fare_per_mile,opportunity_score,significant_10pct,significant_5pct
21,Brooklyn | low_rain | Evening | 0,"[PU_Borough, rain_bucket, PU_time_zone, PU_wee...",99,-1.329704,0.373438,3.698614e-04,0.196488,0.711512,0.695143,17076000,...,7.182259,0.162171,5.582253,7.393982,0.192827,5.344699,-0.032040,24.770484,True,True
13,Brooklyn | high_rain | Evening | 0,"[PU_Borough, rain_bucket, PU_time_zone, PU_wee...",118,1.205177,0.466672,9.808989e-03,0.216864,0.722537,0.667315,19458800,...,6.828948,0.148378,5.251092,7.061207,0.204808,5.157698,-0.051489,22.439954,True,True
77,Queens | no_rain | Evening | 0,"[PU_Borough, rain_bucket, PU_time_zone, PU_wee...",324,-0.912917,0.097509,7.798109e-21,0.270748,0.721704,0.669133,35684800,...,5.537748,0.143940,6.077591,5.606892,0.189259,5.878394,-0.039448,17.832166,True,True
19,Brooklyn | low_rain | Afternoon | 0,"[PU_Borough, rain_bucket, PU_time_zone, PU_wee...",113,-0.914221,0.243899,1.779917e-04,0.400008,0.711260,0.693268,16066800,...,7.220712,0.167937,4.851562,7.359329,0.198172,4.895561,-0.023333,16.981763,True,True
41,Manhattan | high_rain | Morning | 0,"[PU_Borough, rain_bucket, PU_time_zone, PU_wee...",168,-0.867009,0.167403,2.228880e-07,0.291809,0.724367,0.795056,18812400,...,7.485814,0.277675,3.947492,6.414605,0.234902,4.125865,0.103451,16.863812,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43,Manhattan | low_rain | Afternoon | 0,"[PU_Borough, rain_bucket, PU_time_zone, PU_wee...",132,-0.022069,0.192470,9.087122e-01,0.249726,0.711708,0.788150,23790400,...,9.827071,0.276154,4.426141,8.202106,0.237691,4.789001,0.133741,0.436950,False,False
17,Brooklyn | high_rain | Morning | 0,"[PU_Borough, rain_bucket, PU_time_zone, PU_wee...",138,0.018068,0.243864,9.409395e-01,0.107740,0.719267,0.711204,14018800,...,6.476625,0.190619,4.812481,6.494306,0.208392,4.798159,-0.001585,0.334907,False,False
35,Manhattan | high_rain | Afternoon | 0,"[PU_Borough, rain_bucket, PU_time_zone, PU_wee...",168,-0.013133,0.155592,9.327321e-01,0.081272,0.735529,0.798131,33078800,...,10.001208,0.262286,4.341163,8.481500,0.232216,4.636908,0.109573,0.262967,False,False
44,Manhattan | low_rain | Afternoon | 1,"[PU_Borough, rain_bucket, PU_time_zone, PU_wee...",306,0.008586,0.104200,9.343285e-01,0.113914,0.730198,0.816897,50183600,...,11.205515,0.308825,4.105554,8.825120,0.240677,4.548675,0.191613,0.178432,False,False


In [50]:
m_interact = smf.ols(
    formula=f"""
    uber_share_logit
    ~
    + log_rel_lyft_fare_per_mile * C(PU_peak_flag) * C(PU_weekday) * C(PU_Borough)
    """,
    data=uber_df
).fit(cov_type='HC3')

print(m_interact.summary())


                            OLS Regression Results                            
Dep. Variable:       uber_share_logit   R-squared:                       0.572
Model:                            OLS   Adj. R-squared:                  0.571
Method:                 Least Squares   F-statistic:                     1803.
Date:                Wed, 06 May 2026   Prob (F-statistic):               0.00
Time:                        21:23:57   Log-Likelihood:                 352.55
No. Observations:               27567   AIC:                            -641.1
Df Residuals:                   27535   BIC:                            -377.9
Df Model:                          31                                         
Covariance Type:                  HC3                                         
                                                                                                    coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------

### Supply side

In [53]:
base_controls = """
+ C(PU_weekday)
+ C(PU_peak_flag)
+ C(rain_bucket)
+ C(tmin_f_bucket)
+ C(PU_Borough)
"""

In [52]:
uber_df = uber_df.sort_values(['PU_Borough','PU_datetime_hour'])

In [ ]:
uber_df['uber_avg_fare_per_mile']

In [ ]:
## Lag driver pay
uber_df['lag_rel_driver_pay_per_mile'] = (uber_df.groupby('PU_Borough')['log_rel_lyft_driverpay_per_mile'].shift(1))
uber_df['lag_uber_avg_fare_per_mile'] = (uber_df.groupby('PU_Borough')['uber_avg_fare_per_mile'].shift(1))

## Lag total rideshare demand
uber_df['log_total_hvfhv_trips'] = np.log(uber_df['total_hvfhv_trips'] + 1) 
uber_df['lag_log_total_hvfhv'] = (uber_df.groupby('PU_Borough')['log_total_hvfhv_trips'].shift(1))

## Lag take rate
uber_df['lag_log_rel_lyft_take_rate'] = (uber_df.groupby('PU_Borough')['log_rel_lyft_take_rate'].shift(1))
uber_df['lag_uber_avg_take_rate'] = (uber_df.groupby('PU_Borough')['uber_avg_take_rate'].shift(1))

In [69]:
m_s1 = smf.ols(
    formula=f"""
    log_rel_lyft_wait_time
    ~ lag_rel_driver_pay_per_mile
    + lag_log_rel_lyft_take_rate
    + lag_log_total_hvfhv
    {base_controls}
    """,
    data=uber_df
).fit(cov_type='HC3')

print(m_s1.summary())
summary_text_path = OUTPUT_DIR / "m_s1_summary.txt" 
with open(summary_text_path, "w") as f: 
    f.write(m_s1.summary().as_text())

                              OLS Regression Results                              
Dep. Variable:     log_rel_lyft_wait_time   R-squared:                       0.109
Model:                                OLS   Adj. R-squared:                  0.109
Method:                     Least Squares   F-statistic:                     232.9
Date:                    Wed, 06 May 2026   Prob (F-statistic):               0.00
Time:                            22:01:36   Log-Likelihood:                 21623.
No. Observations:                   27563   AIC:                        -4.322e+04
Df Residuals:                       27549   BIC:                        -4.310e+04
Df Model:                              13                                         
Covariance Type:                      HC3                                         
                                        coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------

In [132]:
m_s2 = smf.ols(
    formula=f"""
    uber_avg_wait_time
    ~ lag_uber_avg_fare_per_mile
    + lag_uber_avg_take_rate
    + lag_log_total_hvfhv
    {base_controls}
    """,
    data=uber_df
).fit(cov_type='HC3')

print(m_s2.summary())
summary_text_path = OUTPUT_DIR / "m_s2_summary.txt" 
with open(summary_text_path, "w") as f: 
    f.write(m_s1.summary().as_text())

                            OLS Regression Results                            
Dep. Variable:     uber_avg_wait_time   R-squared:                       0.206
Model:                            OLS   Adj. R-squared:                  0.205
Method:                 Least Squares   F-statistic:                     517.4
Date:                Thu, 07 May 2026   Prob (F-statistic):               0.00
Time:                        03:59:30   Log-Likelihood:                -35381.
No. Observations:               27563   AIC:                         7.079e+04
Df Residuals:                   27549   BIC:                         7.091e+04
Df Model:                          13                                         
Covariance Type:                  HC3                                         
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

In [71]:
## Driver pay x rain
m_s2 = smf.ols(
    formula=f"""
    log_rel_lyft_wait_time
    ~ lag_rel_driver_pay_per_mile
    + lag_rel_driver_pay_per_mile * C(rain_bucket)
    """,
    data=uber_df
).fit(cov_type='HC3')

print(m_s2.summary())
summary_text_path = OUTPUT_DIR / "m_s2_summary.txt" 
with open(summary_text_path, "w") as f: 
    f.write(m_s2.summary().as_text())

                              OLS Regression Results                              
Dep. Variable:     log_rel_lyft_wait_time   R-squared:                       0.027
Model:                                OLS   Adj. R-squared:                  0.027
Method:                     Least Squares   F-statistic:                     109.3
Date:                    Wed, 06 May 2026   Prob (F-statistic):          9.95e-115
Time:                            22:02:17   Log-Likelihood:                 20407.
No. Observations:                   27563   AIC:                        -4.080e+04
Df Residuals:                       27557   BIC:                        -4.075e+04
Df Model:                               5                                         
Covariance Type:                      HC3                                         
                                                             coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------

In [ ]:
## Driver pay x rain
m_s2 = smf.ols(
    formula=f"""
    log_rel_lyft_wait_time
    ~ lag_rel_driver_pay_per_mile
    + lag_rel_driver_pay_per_mile * C(rain_bucket) * C(PU_peak_flag)
    """,
    data=uber_df
).fit(cov_type='HC3')

print(m_s2.summary())
summary_text_path = OUTPUT_DIR / "m_s2_summary.txt" 
with open(summary_text_path, "w") as f: 
    f.write(m_s2.summary().as_text())

                              OLS Regression Results                              
Dep. Variable:     log_rel_lyft_wait_time   R-squared:                       0.036
Model:                                OLS   Adj. R-squared:                  0.035
Method:                     Least Squares   F-statistic:                     84.11
Date:                    Wed, 06 May 2026   Prob (F-statistic):          3.98e-188
Time:                            22:02:00   Log-Likelihood:                 20533.
No. Observations:                   27563   AIC:                        -4.104e+04
Df Residuals:                       27551   BIC:                        -4.094e+04
Df Model:                              11                                         
Covariance Type:                      HC3                                         
                                                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------

In [64]:
## Driver pay x peak hour
m_s2 = smf.ols(
    formula=f"""
    log_rel_lyft_wait_time
    ~ lag_rel_driver_pay_per_mile
    + lag_rel_driver_pay_per_mile * C(PU_peak_flag) * C(PU_Borough)
    """,
    data=uber_df
).fit(cov_type='HC3')

print(m_s2.summary())

                              OLS Regression Results                              
Dep. Variable:     log_rel_lyft_wait_time   R-squared:                       0.096
Model:                                OLS   Adj. R-squared:                  0.095
Method:                     Least Squares   F-statistic:                     193.3
Date:                    Wed, 06 May 2026   Prob (F-statistic):               0.00
Time:                            21:50:17   Log-Likelihood:                 21417.
No. Observations:                   27563   AIC:                        -4.280e+04
Df Residuals:                       27547   BIC:                        -4.267e+04
Df Model:                              15                                         
Covariance Type:                      HC3                                         
                                                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------